In [ ]:
# Run once on first open. Skip / comment out after the first successful run.
!pip3 install --quiet numpy pandas matplotlib seaborn plotly statsmodels

# Round 5 — Combinatorial Correlation & Basket Search

This notebook is dedicated to **finding linear combinations of products that mean-revert**, exhaustively across the 50 Round-5 products. It complements `round_analysis.ipynb` (single-product / pair EDA) by going wider and deeper into multi-product baskets.

## Search ladder (each section builds on the previous)

| § | Search | What we find |
|---|---|---|
| §2 | Pearson + Spearman — all 1 ↔ 1 | which products move together (level + returns) |
| §3 | 1 ↔ 1 cointegration — all C(50,2)=1225 pairs | best raw pairs by Engle-Granger / ADF |
| §4 | 1 ↔ 1 **integer** hedge ratios | clean ratios like `A − 2·B`, `A + 3·B` |
| §5 | 1 ↔ many continuous — exhaustive K=1..4 | best real-weight basket per target |
| §6 | 1 ↔ many **integer** weights | "true" baskets like `target ≈ 3·C + 2·Y + Z` |
| §7 | Group-vs-group composite indices | macro-level relationships between the 10 announced groups |
| §8 | Pure integer linear combinations | stationary baskets `c₁·P₁ + … + cₖ·Pₖ ≈ const` (k ∈ {2..5}) — no notion of target/hedge |
| §9 | Cross-group focus | best basket per (target_group, basket_group) cell |
| §10 | Dashboard + exports | top results CSVs and residual plots |

## Why this scales

Naively regressing each subset of K products on a target requires solving the OLS system for every subset — for K=4 that's ≈ 10.6 M fits.

We avoid this with a **Gram-matrix trick**:

1. Mean-center the price panel `X` (T × 50) → `Xc`.
2. Compute the 50×50 covariance matrix `G = Xcᵀ Xc / T` once.
3. For any subset `S` and target `t`, the OLS residual variance is
    `var_resid = G[t,t] − G[t,S] · G[S,S]⁻¹ · G[S,t]`
    using only the small submatrices of `G`. We never touch the full data again until we want to score the surviving top-N with ADF.

The same trick gives us **integer** coefficient evaluation: residual variance for integer vector `c` against subset `S` is just `cᵀ G[S,S] c − 2 cᵀ G[S,t] + G[t,t]`, also a tiny matrix multiply.

## Knobs you can tune in §1

- `K_MAX` — max basket size. 3 is fast, 4 is ~minutes, 5 only if you have ~10 GB RAM and patience.
- `INT_COEF_MIN/MAX` — integer coefficient range, default ±5.
- `TOP_N_PER_K` — only the top-N continuous baskets per K get the (slow) ADF / half-life pass.
- `SUBSAMPLE_FOR_ADF` — random tick subsample for ADF when the full panel is huge.


## §1 — Setup, config, helpers

In [ ]:
import math
import time
import warnings
from pathlib import Path
from itertools import combinations, product as iproduct

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import adfuller, coint
from IPython.display import display, Markdown

%matplotlib inline
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", font_scale=0.9)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

# ============ CONFIG ============
DATA_DIR = Path("ROUND_5")
ROUND    = 5
OUT_DIR  = Path("combination_search_outputs")
OUT_DIR.mkdir(exist_ok=True)
# ================================

PRODUCT_GROUPS = {
    "galaxy_sounds": ["GALAXY_SOUNDS_DARK_MATTER", "GALAXY_SOUNDS_BLACK_HOLES",
                      "GALAXY_SOUNDS_PLANETARY_RINGS", "GALAXY_SOUNDS_SOLAR_WINDS",
                      "GALAXY_SOUNDS_SOLAR_FLAMES"],
    "sleep_pods":    ["SLEEP_POD_SUEDE", "SLEEP_POD_LAMB_WOOL", "SLEEP_POD_POLYESTER",
                      "SLEEP_POD_NYLON", "SLEEP_POD_COTTON"],
    "microchips":    ["MICROCHIP_CIRCLE", "MICROCHIP_OVAL", "MICROCHIP_SQUARE",
                      "MICROCHIP_RECTANGLE", "MICROCHIP_TRIANGLE"],
    "pebbles":       ["PEBBLES_XS", "PEBBLES_S", "PEBBLES_M", "PEBBLES_L", "PEBBLES_XL"],
    "robots":        ["ROBOT_VACUUMING", "ROBOT_MOPPING", "ROBOT_DISHES",
                      "ROBOT_LAUNDRY", "ROBOT_IRONING"],
    "uv_visors":     ["UV_VISOR_YELLOW", "UV_VISOR_AMBER", "UV_VISOR_ORANGE",
                      "UV_VISOR_RED", "UV_VISOR_MAGENTA"],
    "translators":   ["TRANSLATOR_SPACE_GRAY", "TRANSLATOR_ASTRO_BLACK",
                      "TRANSLATOR_ECLIPSE_CHARCOAL", "TRANSLATOR_GRAPHITE_MIST",
                      "TRANSLATOR_VOID_BLUE"],
    "panels":        ["PANEL_1X2", "PANEL_2X2", "PANEL_1X4", "PANEL_2X4", "PANEL_4X4"],
    "oxygen_shakes": ["OXYGEN_SHAKE_MORNING_BREATH", "OXYGEN_SHAKE_EVENING_BREATH",
                      "OXYGEN_SHAKE_MINT", "OXYGEN_SHAKE_CHOCOLATE",
                      "OXYGEN_SHAKE_GARLIC"],
    "snackpacks":    ["SNACKPACK_CHOCOLATE", "SNACKPACK_VANILLA", "SNACKPACK_PISTACHIO",
                      "SNACKPACK_STRAWBERRY", "SNACKPACK_RASPBERRY"],
}
ALL_PRODUCTS     = [p for ms in PRODUCT_GROUPS.values() for p in ms]
PRODUCT_TO_GROUP = {p: g for g, ms in PRODUCT_GROUPS.items() for p in ms}
GROUP_NAMES      = list(PRODUCT_GROUPS.keys())

# --- Combinatorial knobs ---
K_MAX             = 4       # max basket size for §5/§6 (4 ≈ minutes; 5 is slow)
INT_COEF_MIN      = -5
INT_COEF_MAX      =  5
TOP_N_PER_K       = 200     # surviving baskets per K that get ADF / half-life
TOP_DISPLAY       = 40      # rows shown in printed tables
TOP_PER_TARGET_K  = 50      # keep at most this many subsets per (target, K) before ADF
SUBSAMPLE_FOR_ADF = 20000   # random rows for ADF (None = all)
RANDOM_SEED       = 42

# --- Trading-relevant defaults (used in dashboard PnL gate) ---
WINDOW_FAST       = 200
Z_ENTRY           = 2.0
Z_EXIT            = 0.5

np.random.seed(RANDOM_SEED)


In [ ]:
# ─── stat helpers ───
def adf_p(series, subsample=None):
    if subsample is None:
        subsample = globals().get('SUBSAMPLE_FOR_ADF', 20000)
    """ADF p-value. Optionally subsample for speed."""
    s = pd.Series(series).dropna().values
    if len(s) < 30 or np.std(s) == 0:
        return np.nan
    if subsample and len(s) > subsample:
        idx = np.linspace(0, len(s) - 1, subsample).astype(int)
        s = s[idx]
    try:
        return float(adfuller(s, autolag="AIC")[1])
    except Exception:
        return np.nan

def half_life(series):
    """OU half-life in ticks. NaN if non-reverting."""
    s = pd.Series(series).dropna().values
    if len(s) < 50:
        return np.nan
    x_lag = s[:-1]
    dx    = np.diff(s)
    X     = np.column_stack([np.ones_like(x_lag), x_lag])
    try:
        b = np.linalg.lstsq(X, dx, rcond=None)[0][1]
    except Exception:
        return np.nan
    return float(np.log(2) / -b) if (np.isfinite(b) and b < 0) else np.nan

def fmt_basket(prods, coefs, max_terms=8):
    """Format an integer/float basket as e.g. '3*A − 2*B + C'."""
    parts = []
    for p, c in zip(prods, coefs):
        if abs(c) < 1e-9:
            continue
        sign = "+" if c > 0 else "−"
        mag  = abs(c)
        mag_s = f"{int(round(mag))}" if abs(mag - round(mag)) < 1e-6 else f"{mag:.3f}"
        if mag_s == "1":
            parts.append(f"{sign} {p}")
        else:
            parts.append(f"{sign} {mag_s}*{p}")
    if not parts:
        return "(zero)"
    out = " ".join(parts)
    if out.startswith("+ "):
        out = out[2:]
    return out[:200]

def signed_basket_sum(X, basket_idx, coefs):
    """X[:, basket_idx] @ coefs as a 1-D residual / spread series."""
    return X[:, basket_idx] @ np.asarray(coefs, dtype=float)


## §2 — Data loading and aligned mid panel

Load all `prices_round_5_day_*.csv` files, concatenate, build a global time index `t = day · 1e6 + timestamp`, fill empty-book mids, then **pivot to a wide aligned panel** `mid_aligned` of shape `(T, 50)` where every column is a product mid and every row is a tick where *every* product traded.

The aligned panel is the input to every numerical search downstream. We also pre-build:

- `X` — float numpy view, shape `(T, 50)`, in `ALL_PRODUCTS` order
- `Xc` — mean-centered version of `X`
- `G` — covariance matrix `Xc.T @ Xc / T`, shape `(50, 50)`
- `var` — the diagonal of `G`, length 50

`G` is the workhorse. With it, every OLS fit becomes an O(K³) operation independent of `T`.


In [ ]:
import re

def load_round5(data_dir: Path) -> pd.DataFrame:
    files = sorted(data_dir.glob("prices_round_*_day_*.csv"))
    if not files:
        raise FileNotFoundError(f"No prices CSVs in {data_dir}")
    frames = []
    for f in files:
        df = pd.read_csv(f, sep=";")
        if "day" not in df.columns:
            m = re.search(r"day_(-?\d+)", f.stem)
            if m:
                df["day"] = int(m.group(1))
        frames.append(df)
    prices = pd.concat(frames, ignore_index=True)
    prices["t"] = prices["day"].astype(int) * 1_000_000 + prices["timestamp"].astype(int)
    prices.loc[prices["mid_price"] == 0, "mid_price"] = np.nan
    prices["mid_price"] = prices.groupby("product")["mid_price"].ffill()
    return prices

prices = load_round5(DATA_DIR)
print(f"Loaded {len(prices):,} rows across days {sorted(prices['day'].unique())}")
print(f"Distinct products: {prices['product'].nunique()}")
missing = [p for p in ALL_PRODUCTS if p not in set(prices["product"])]
if missing:
    print(f"⚠ Missing from data: {missing}")

mid = prices.pivot_table(index="t", columns="product", values="mid_price", aggfunc="last")
present = [p for p in ALL_PRODUCTS if p in mid.columns]
mid = mid[present].sort_index()
mid_aligned = mid.dropna()
T, P = mid_aligned.shape
print(f"\nAligned panel: T = {T:,} ticks × P = {P} products")

# Numpy backing arrays + Gram matrix
X       = mid_aligned.values.astype(np.float64)
columns = list(mid_aligned.columns)
mean_X  = X.mean(axis=0)
Xc      = X - mean_X
G       = (Xc.T @ Xc) / T
var     = np.diag(G).copy()
std     = np.sqrt(var)

print(f"Gram matrix G shape: {G.shape}")
print(f"Per-product std: min={std.min():.3f}, median={np.median(std):.3f}, max={std.max():.3f}")


## §3 — Pearson correlations (all C(50,2) = 1225 pairs)

Three rankings of the same 1225 pairs:

1. **Top 40 by `|pearson_level|`** — raw mid-price correlation. High here alone is suspect (two products both drifting up will look correlated even if unrelated tick-by-tick).
2. **Top 40 by `|pearson_ret|`** — returns / first-difference correlation. The honest signal: do the products actually move together at the tick level?
3. **Top 40 by joint score** — geometric mean `√(|pearson_level| · |pearson_ret|)`. Surfaces pairs where **both** are high — the strongest pair-trade candidates (shared trend AND co-move tick-by-tick).

Spearman is skipped — for tightly-traded products with continuous prices, Pearson and Spearman agree closely and Spearman just doubles the table count.


In [ ]:
rets = mid_aligned.diff().iloc[1:]

corr_level = mid_aligned.corr(method="pearson")
corr_ret   = rets.corr(method="pearson")

def pair_long(C: pd.DataFrame) -> pd.DataFrame:
    rows = []
    cols = list(C.columns)
    for i, a in enumerate(cols):
        for j, b in enumerate(cols):
            if i >= j:
                continue
            rows.append({"a": a, "b": b, "corr": C.iat[i, j]})
    return pd.DataFrame(rows)

p_level = pair_long(corr_level).rename(columns={"corr": "pearson_level"})
p_ret   = pair_long(corr_ret).rename(columns={"corr": "pearson_ret"})
both = p_level.merge(p_ret, on=["a", "b"], how="inner")

both["abs_level"]   = both["pearson_level"].abs()
both["abs_ret"]     = both["pearson_ret"].abs()
both["joint_score"] = np.sqrt(both["abs_level"] * both["abs_ret"])
both["group_a"]     = both["a"].map(PRODUCT_TO_GROUP)
both["group_b"]     = both["b"].map(PRODUCT_TO_GROUP)
both["cross_group"] = both["group_a"] != both["group_b"]

both.to_csv(OUT_DIR / "pair_correlations.csv", index=False)

cols_show = ["a", "b", "group_a", "group_b", "cross_group",
             "pearson_level", "pearson_ret", "joint_score"]

print("=== Top 40 pairs by |pearson_level| ===")
display(both.sort_values("abs_level", ascending=False).head(40)[cols_show].round(4))

print("\n=== Top 40 pairs by |pearson_ret| ===")
display(both.sort_values("abs_ret", ascending=False).head(40)[cols_show].round(4))

print("\n=== Top 40 pairs by joint score (both level AND returns high) ===")
display(both.sort_values("joint_score", ascending=False).head(40)[cols_show].round(4))

print("\n=== Top 40 CROSS-GROUP pairs by joint score ===")
display(both.query("cross_group").sort_values("joint_score", ascending=False).head(40)[cols_show].round(4))


In [ ]:
# Group-blocked heatmaps (level + returns Pearson)
ordered = list(mid_aligned.columns)
group_bounds = []
cum = 0
for g in GROUP_NAMES:
    in_panel = [p for p in PRODUCT_GROUPS[g] if p in ordered]
    cum += len(in_panel)
    group_bounds.append(cum)

fig, axs = plt.subplots(1, 2, figsize=(20, 10))
for ax, C, title in [(axs[0], corr_level_pearson, "Pearson — level (group-blocked)"),
                     (axs[1], corr_ret_pearson,   "Pearson — returns (group-blocked)")]:
    sns.heatmap(C, cmap="coolwarm", center=0, vmin=-1, vmax=1, square=True,
                annot=False, cbar_kws={"shrink": 0.6}, ax=ax)
    for b in group_bounds[:-1]:
        ax.axhline(b, color="black", lw=1.0)
        ax.axvline(b, color="black", lw=1.0)
    ax.set_title(title)
    ax.tick_params(axis="x", labelsize=5, rotation=90)
    ax.tick_params(axis="y", labelsize=5)
plt.tight_layout(); plt.show()


### §3.1 — Cross-group focus

Within-group correlation is structural (5 variants of the same item move together — that's expected and not an edge). The interesting cells are **off-diagonal**: pairs where products from *different* announced groups are linked.

This block strips out within-group pairs entirely and produces:

1. Top 40 cross-group pairs by joint score, by `|pearson_ret|`, by `|pearson_level|`.
2. Best cross-group pair per `(group_a, group_b)` cell — a 10×10 matrix view of where the cross-group co-movement lives.
3. A returns-correlation heatmap with within-group blocks masked, so cross-group reds stand out visually.


In [ ]:
cross = both.query("cross_group").copy()

cols_show = ["a", "b", "group_a", "group_b",
             "pearson_level", "pearson_ret", "joint_score"]

print(f"Total cross-group pairs: {len(cross):,} of {len(both):,} all pairs\n")

print("=== Top 40 CROSS-GROUP pairs by joint score ===")
display(cross.sort_values("joint_score", ascending=False).head(40)[cols_show].round(4))

print("\n=== Top 40 CROSS-GROUP pairs by |pearson_ret| ===")
display(cross.sort_values("abs_ret", ascending=False).head(40)[cols_show].round(4))

print("\n=== Top 40 CROSS-GROUP pairs by |pearson_level| ===")
display(cross.sort_values("abs_level", ascending=False).head(40)[cols_show].round(4))

# Best cross-group pair per (group_a, group_b) cell — symmetric, so canonicalise the pair
cross_canon = cross.copy()
cross_canon["g_lo"] = cross_canon[["group_a", "group_b"]].min(axis=1)
cross_canon["g_hi"] = cross_canon[["group_a", "group_b"]].max(axis=1)
best_per_cell = (cross_canon.sort_values("joint_score", ascending=False)
                            .groupby(["g_lo", "g_hi"], as_index=False).first()
                            .sort_values("joint_score", ascending=False))

print("\n=== Best cross-group pair per (group, group) cell — sorted by joint score ===")
display(best_per_cell[["g_lo", "g_hi", "a", "b", "pearson_level", "pearson_ret", "joint_score"]]
        .head(45).round(4))

best_per_cell.to_csv(OUT_DIR / "cross_group_best_pair_per_cell.csv", index=False)

# Heatmap of best joint score per (group, group) cell
heat = (best_per_cell.pivot(index="g_lo", columns="g_hi", values="joint_score")
                     .reindex(index=GROUP_NAMES, columns=GROUP_NAMES))
fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(heat, annot=True, fmt=".2f", cmap="magma",
            cbar_kws={"label": "best joint score in cell"}, ax=ax)
ax.set_title("Cross-group co-movement: best joint score per (group, group) cell")
plt.tight_layout(); plt.show()

# Returns-correlation heatmap with within-group blocks masked
ordered_p = [p for g in GROUP_NAMES for p in PRODUCT_GROUPS[g] if p in mid_aligned.columns]
C = corr_ret.loc[ordered_p, ordered_p].copy()
mask = pd.DataFrame(False, index=C.index, columns=C.columns)
for g, members in PRODUCT_GROUPS.items():
    in_panel = [p for p in members if p in C.columns]
    for i in in_panel:
        for j in in_panel:
            mask.at[i, j] = True

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(C, mask=mask, cmap="coolwarm", center=0, vmin=-1, vmax=1,
            square=True, annot=False, cbar_kws={"shrink": 0.6}, ax=ax)
group_bounds = []
cum = 0
for g in GROUP_NAMES:
    in_panel = [p for p in PRODUCT_GROUPS[g] if p in ordered_p]
    cum += len(in_panel)
    group_bounds.append(cum)
for b in group_bounds[:-1]:
    ax.axhline(b, color="black", lw=1.0)
    ax.axvline(b, color="black", lw=1.0)
ax.set_title("Returns correlation — within-group blocks masked (cross-group only)")
ax.tick_params(axis="x", labelsize=5, rotation=90)
ax.tick_params(axis="y", labelsize=5)
plt.tight_layout(); plt.show()


## §4 — 1 ↔ 1 cointegration scan (all 1225 pairs)

For every pair `(a, b)`:

1. OLS hedge ratio `h = cov(a,b) / var(b)` (closed form via the Gram matrix).
2. Spread `s = a − h·b`, residual std ratio = `√(1 − corr²)`.
3. **Engle-Granger** cointegration p-value (`coint`) — uses the right critical values for an OLS-fitted spread.
4. ADF p-value on the spread.
5. OU half-life.

EG is run on every pair (1225 × ≈ 30 ms ≈ 30 s). We sort by EG p ascending → the strongest cointegrations float to the top.


In [ ]:
# Two-stage scan: cheap closed-form for ALL 1225 pairs, then EG + ADF on top by |level_corr|.
COINT_TOP = 300   # number of candidates for the slow tests
COINT_MAXLAG = 5  # fixed lag — skips internal AIC autolag (≈5× speedup per call)

t0 = time.time()
fast_rows = []
for i, j in combinations(range(P), 2):
    if var[j] == 0 or var[i] == 0:
        continue
    h        = G[i, j] / var[j]
    rvar     = var[i] - h * G[i, j]
    lvl_corr = G[i, j] / np.sqrt(var[i] * var[j])
    fast_rows.append({
        "i": i, "j": j,
        "a": columns[i], "b": columns[j],
        "group_a": PRODUCT_TO_GROUP[columns[i]],
        "group_b": PRODUCT_TO_GROUP[columns[j]],
        "cross_group":     PRODUCT_TO_GROUP[columns[i]] != PRODUCT_TO_GROUP[columns[j]],
        "hedge_ratio":     float(h),
        "level_corr":      float(lvl_corr),
        "abs_level_corr":  float(abs(lvl_corr)),
        "resid_std_ratio": float(np.sqrt(max(rvar, 0) / var[i])),
    })
fast_df = pd.DataFrame(fast_rows)
print(f"Stage 1 (closed-form, all {len(fast_df)} pairs): {time.time()-t0:.2f}s")

# Stage 2: coint + ADF only for top COINT_TOP by |level_corr|
t1 = time.time()
candidates = fast_df.sort_values("abs_level_corr", ascending=False).head(COINT_TOP).copy()

def coint_p_fast(xi, xj):
    try:
        return float(coint(xi, xj, trend="c", maxlag=COINT_MAXLAG, autolag=None)[1])
    except Exception:
        return np.nan

def adf_p_fast(spread):
    s = pd.Series(spread).dropna().values
    if SUBSAMPLE_FOR_ADF and len(s) > SUBSAMPLE_FOR_ADF:
        s = s[np.linspace(0, len(s) - 1, SUBSAMPLE_FOR_ADF).astype(int)]
    if len(s) < 30 or np.std(s) == 0:
        return np.nan
    try:
        return float(adfuller(s, maxlag=COINT_MAXLAG, autolag=None)[1])
    except Exception:
        return np.nan

eg_ps, adf_ps, hls = [], [], []
for _, row in candidates.iterrows():
    i, j   = int(row["i"]), int(row["j"])
    h      = row["hedge_ratio"]
    spread = X[:, i] - h * X[:, j]
    eg_ps.append(coint_p_fast(X[:, i], X[:, j]))
    adf_ps.append(adf_p_fast(spread))
    hls.append(half_life(spread))
candidates["eg_p"]      = eg_ps
candidates["adf_p"]     = adf_ps
candidates["half_life"] = hls
print(f"Stage 2 (EG + ADF, top {len(candidates)} candidates): {time.time()-t1:.1f}s")

# Merge: any pair not tested gets NaN for the heavy stats
pair_table = (fast_df.merge(
        candidates[["a", "b", "eg_p", "adf_p", "half_life"]],
        on=["a", "b"], how="left")
    .drop(columns=["i", "j", "abs_level_corr"])
    .sort_values("eg_p", na_position="last")
    .reset_index(drop=True))
pair_table.to_csv(OUT_DIR / "pair_cointegration.csv", index=False)
print(f"Total: {time.time()-t0:.1f}s, full table size: {len(pair_table)}")

print("\n=== Top pairs by EG p-value (only candidates that ran the test) ===")
display(pair_table.dropna(subset=["eg_p"]).head(TOP_DISPLAY).round(4))

print("\n=== Top CROSS-GROUP pairs by EG p-value ===")
display(pair_table.dropna(subset=["eg_p"]).query("cross_group").head(TOP_DISPLAY).round(4))


## §5 — 1 ↔ 1 integer hedge search

Real-valued hedge ratios are messy in practice (you can't trade 1.7 units of a product). We search the full integer grid `[INT_COEF_MIN .. INT_COEF_MAX] \ {0}` for every pair from §4 and pick the integer ratio that minimizes spread variance:

`var(a − k·b) = var(a) − 2·k·cov(a,b) + k²·var(b)`

(closed-form via the Gram matrix — instant for 1225 pairs × 11 integers).

We then run ADF / half-life on the integer-hedge spread and rank.


In [ ]:
int_grid = [k for k in range(INT_COEF_MIN, INT_COEF_MAX + 1) if k != 0]
int_rows = []
for _, r in pair_table.iterrows():
    a, b = r["a"], r["b"]
    i, j = columns.index(a), columns.index(b)
    best_k, best_v = None, np.inf
    for k in int_grid:
        v = var[i] - 2 * k * G[i, j] + k ** 2 * var[j]
        if v < best_v:
            best_v, best_k = v, k
    spread = X[:, i] - best_k * X[:, j]
    int_rows.append({
        "a": a, "b": b,
        "group_a": r["group_a"], "group_b": r["group_b"],
        "cross_group":          r["cross_group"],
        "ols_h":                r["hedge_ratio"],
        "int_h":                int(best_k),
        "ols_resid_std_ratio":  r["resid_std_ratio"],
        "int_resid_std_ratio":  float(np.sqrt(max(best_v, 0) / var[i])),
        "int_resid_inflation":  float(np.sqrt(max(best_v, 0) / var[i]) / max(r["resid_std_ratio"], 1e-9)),
        "int_adf_p":            adf_p(spread),
        "int_half_life":        half_life(spread),
    })
int_pair_table = pd.DataFrame(int_rows).sort_values("int_adf_p").reset_index(drop=True)
int_pair_table.to_csv(OUT_DIR / "pair_integer_hedge.csv", index=False)

print("\n=== Top integer-hedge pairs by ADF p ===")
display(int_pair_table.head(TOP_DISPLAY).round(4))

print("\n=== Integer pairs where rounding is nearly free (inflation < 1.05) ===")
display(int_pair_table.query("int_resid_inflation < 1.05")
                      .sort_values("int_adf_p")
                      .head(TOP_DISPLAY).round(4))


## §6 — 1 ↔ many continuous basket search (K = 1..K_MAX)

For each of the 50 products as **target**, we exhaustively try every subset of size `k = 1, 2, …, K_MAX` from the other 49 products as a hedge basket.

Per subset we use the Gram trick to compute residual variance in O(k³):

```
β  = G[S,S]⁻¹ · G[S,t]
var_resid = G[t,t] − β · G[S,t]
```

We keep the **top `TOP_PER_TARGET_K` baskets per target per K** (sorted by R²), then run ADF / half-life only on the global **top `TOP_N_PER_K` per K**.

Approximate scale at K_MAX=4:

- K=1: 50 × 49 = 2 450 fits
- K=2: 50 × C(49,2) = 58 800
- K=3: 50 × C(49,3) = 921 200
- K=4: 50 × C(49,4) = 10 593 800

Each fit is a tiny matrix solve, so the whole sweep usually takes seconds for K ≤ 3, a few minutes for K=4.


In [ ]:
def basket_resid_var(target_idx, S):
    """Residual variance of OLS regressing X[:,target] on X[:,S] using Gram trick."""
    Gss = G[np.ix_(S, S)]
    Gst = G[S, target_idx]
    try:
        beta = np.linalg.solve(Gss, Gst)
    except np.linalg.LinAlgError:
        return np.inf, None
    rvar = var[target_idx] - beta @ Gst
    if rvar < -1e-9:
        return np.inf, None
    return max(rvar, 0.0), beta

def reconstruct_residual(target_idx, S, beta):
    """Time-series residual: y − (intercept + Xs·β). Centered residual via Xc."""
    return Xc[:, target_idx] - Xc[:, S] @ beta

def all_subsets_search(k_max=K_MAX, top_per_target=TOP_PER_TARGET_K):
    """Exhaustive search across all (target, K-subset) for K=1..k_max. Returns dict[K] -> DataFrame."""
    results = {}
    for k in range(1, k_max + 1):
        t0 = time.time()
        per_target = {ti: [] for ti in range(P)}
        n_eval = 0
        for ti in range(P):
            cands = [j for j in range(P) if j != ti]
            for combo in combinations(cands, k):
                S = list(combo)
                rvar, beta = basket_resid_var(ti, S)
                if not np.isfinite(rvar) or beta is None:
                    continue
                per_target[ti].append((S, beta.copy(), rvar))
                n_eval += 1
            per_target[ti].sort(key=lambda r: r[2])
            per_target[ti] = per_target[ti][:top_per_target]
        elapsed = time.time() - t0
        rate = n_eval / max(elapsed, 1e-6)
        rows = []
        for ti, locals_ in per_target.items():
            for S, beta, rvar in locals_:
                target = columns[ti]
                basket = [columns[s] for s in S]
                groups = sorted({PRODUCT_TO_GROUP[p] for p in basket})
                rows.append({
                    "target":         target,
                    "target_group":   PRODUCT_TO_GROUP[target],
                    "k":              k,
                    "basket":         basket,
                    "basket_groups":  groups,
                    "n_groups":       len(groups),
                    "cross_group":    PRODUCT_TO_GROUP[target] not in groups,
                    "ols_beta":       beta.tolist(),
                    "resid_var":      float(rvar),
                    "r_squared":      float(1 - rvar / var[ti]) if var[ti] > 0 else np.nan,
                    "S_idx":          S,
                    "target_idx":     ti,
                })
        df = pd.DataFrame(rows).sort_values("resid_var").reset_index(drop=True)
        results[k] = df
        print(f"K={k}: {n_eval:,} evals, {elapsed:.1f}s ({rate:.0f}/s), kept {len(df):,} rows")
    return results

print("Running exhaustive subset search ...")
basket_results = all_subsets_search(K_MAX, TOP_PER_TARGET_K)


In [ ]:
def attach_adf(df_k, top_n=TOP_N_PER_K):
    """Run ADF + half-life only on the top_n by resid_var."""
    if len(df_k) == 0:
        return df_k
    df_top = df_k.sort_values("resid_var").head(top_n).copy().reset_index(drop=True)
    adfs, hls = [], []
    for _, row in df_top.iterrows():
        beta  = np.asarray(row["ols_beta"])
        S     = row["S_idx"]
        ti    = row["target_idx"]
        resid = reconstruct_residual(ti, S, beta)
        adfs.append(adf_p(resid))
        hls.append(half_life(resid))
    df_top["adf_p"]      = adfs
    df_top["half_life"]  = hls
    df_top["basket_str"] = df_top.apply(
        lambda r: fmt_basket([r["target"]] + r["basket"], [1.0] + list(-np.array(r["ols_beta"]))),
        axis=1)
    return df_top.sort_values("adf_p").reset_index(drop=True)

basket_top = {k: attach_adf(df, TOP_N_PER_K) for k, df in basket_results.items()}
for k, df in basket_top.items():
    df.to_csv(OUT_DIR / f"basket_continuous_K{k}.csv", index=False)
    print(f"\n=== Top continuous baskets — K = {k} (sorted by ADF p) ===")
    display(df.head(TOP_DISPLAY)[
        ["target", "target_group", "k", "basket_str", "n_groups",
         "cross_group", "r_squared", "adf_p", "half_life"]
    ].round(4))


## §7 — 1 ↔ many integer-coefficient basket search

This is the section that finds **clean** baskets like `target ≈ 3·C + 2·Y + Z`.

Procedure per surviving continuous basket from §6:

1. Round the OLS β to nearest integer in `[INT_COEF_MIN, INT_COEF_MAX]`.
2. Search a 3ᵏ grid around the rounded vector (each component ∈ {nearest−1, nearest, nearest+1}, clipped to `[INT_COEF_MIN, INT_COEF_MAX]`).
3. Pick the integer vector minimizing residual variance via the closed form `cᵀG[S,S]c − 2·cᵀG[S,t] + G[t,t]`.
4. Optionally also try a wider search (full grid) for tiny baskets (k ≤ 2).

Then run ADF + half-life only on the survivors with the smallest residual variance ratio.


In [ ]:
def integer_var(c_int, target_idx, S):
    """Residual variance of integer combo: var(y - X[:,S]·c_int)."""
    c = np.asarray(c_int, dtype=float)
    return var[target_idx] - 2 * c @ G[S, target_idx] + c @ G[np.ix_(S, S)] @ c

def integer_search_local(beta, target_idx, S, radius=1):
    """3^k grid around np.round(beta); returns (best_c, best_var)."""
    near = np.clip(np.round(beta).astype(int), INT_COEF_MIN, INT_COEF_MAX)
    best_c, best_v = None, np.inf
    deltas = list(range(-radius, radius + 1))
    for d in iproduct(deltas, repeat=len(beta)):
        c = np.clip(near + np.array(d), INT_COEF_MIN, INT_COEF_MAX)
        if (c == 0).all():
            continue
        v = integer_var(c, target_idx, S)
        if v < best_v:
            best_v, best_c = v, c.copy()
    return best_c, best_v

def integer_search_full(target_idx, S):
    """Exhaustive integer grid in [INT_COEF_MIN..INT_COEF_MAX]^k, k small."""
    best_c, best_v = None, np.inf
    rng = list(range(INT_COEF_MIN, INT_COEF_MAX + 1))
    for c in iproduct(rng, repeat=len(S)):
        c_arr = np.asarray(c, dtype=int)
        if (c_arr == 0).all():
            continue
        v = integer_var(c_arr, target_idx, S)
        if v < best_v:
            best_v, best_c = v, c_arr.copy()
    return best_c, best_v

def integer_baskets_per_K(df_top_continuous, full_grid_k_max=2):
    rows = []
    for _, row in df_top_continuous.iterrows():
        ti    = row["target_idx"]
        S     = row["S_idx"]
        beta  = np.asarray(row["ols_beta"])
        if len(S) <= full_grid_k_max:
            c_full, v_full   = integer_search_full(ti, S)
        else:
            c_full, v_full   = None, np.inf
        c_local, v_local     = integer_search_local(beta, ti, S, radius=1)
        if c_full is not None and v_full < v_local:
            c_int, v_int = c_full, v_full
        else:
            c_int, v_int = c_local, v_local
        if c_int is None:
            continue
        target = columns[ti]
        basket = [columns[s] for s in S]
        spread = X[:, ti] - X[:, S] @ c_int.astype(float)
        rows.append({
            "target":              target,
            "target_group":        PRODUCT_TO_GROUP[target],
            "k":                   len(S),
            "basket":              basket,
            "basket_groups":       sorted({PRODUCT_TO_GROUP[p] for p in basket}),
            "ols_beta":            beta.tolist(),
            "int_coefs":           [int(c) for c in c_int],
            "int_basket_str":      fmt_basket([target] + basket, [1.0] + list(-c_int.astype(float))),
            "ols_resid_std_ratio": float(np.sqrt(max(row["resid_var"], 0) / var[ti])),
            "int_resid_std_ratio": float(np.sqrt(max(v_int, 0) / var[ti])),
            "int_resid_inflation": float(np.sqrt(max(v_int, 0) / max(row["resid_var"], 1e-12))),
            "int_adf_p":           adf_p(spread),
            "int_half_life":       half_life(spread),
            "n_groups":            len({PRODUCT_TO_GROUP[p] for p in basket}),
            "cross_group":         PRODUCT_TO_GROUP[target] not in {PRODUCT_TO_GROUP[p] for p in basket},
        })
    return pd.DataFrame(rows).sort_values("int_adf_p").reset_index(drop=True)

basket_int = {}
for k, df_top in basket_top.items():
    print(f"\n--- Integer search for K = {k} on top {len(df_top)} continuous baskets ---")
    if len(df_top) == 0:
        continue
    df_int = integer_baskets_per_K(df_top)
    basket_int[k] = df_int
    df_int.to_csv(OUT_DIR / f"basket_integer_K{k}.csv", index=False)
    display(df_int.head(TOP_DISPLAY)[
        ["target", "target_group", "k", "int_basket_str", "n_groups",
         "cross_group", "int_resid_std_ratio", "int_resid_inflation",
         "int_adf_p", "int_half_life"]
    ].round(4))


## §8 — Group ↔ group composite-index analysis

Each group reduced to a single time-series (its **composite index** = z-scored mean of member mids), then we run the same hierarchy on the 10×10 problem:

- Pearson + Spearman correlation between group indices.
- Pairwise OLS hedge + integer hedge on indices.
- Cointegration p-values across all C(10,2) = 45 group pairs.
- The basket-search machinery from §6/§7 applied at the group level (cheap; trivial subsets).


In [ ]:
def group_index(g):
    members = [p for p in PRODUCT_GROUPS[g] if p in mid_aligned.columns]
    if len(members) < 2:
        return None
    sub = mid_aligned[members]
    z = (sub - sub.mean()) / sub.std(ddof=0).replace(0, np.nan)
    z = z.dropna(axis=1)
    return z.mean(axis=1)

group_idx_df = pd.DataFrame({g: group_index(g) for g in GROUP_NAMES if group_index(g) is not None})
print(f"Built group indices: {list(group_idx_df.columns)}  ({len(group_idx_df)} ticks)")
group_idx_df.to_csv(OUT_DIR / "group_indices.csv")

# Pearson / Spearman + cointegration table
gi_corr_p = group_idx_df.corr(method="pearson")
gi_corr_s = group_idx_df.corr(method="spearman")
print("\n=== Group-index correlation (Pearson) ==="); display(gi_corr_p.round(3))
print("\n=== Group-index correlation (Spearman) ==="); display(gi_corr_s.round(3))

g_cols = list(group_idx_df.columns)
GX     = group_idx_df.values
mean_GX = GX.mean(axis=0)
GXc     = GX - mean_GX
G_g     = (GXc.T @ GXc) / len(GX)
var_g   = np.diag(G_g).copy()

group_pair_rows = []
for i, j in combinations(range(len(g_cols)), 2):
    if var_g[j] == 0: continue
    h_ols  = G_g[i, j] / var_g[j]
    rvar   = var_g[i] - h_ols * G_g[i, j]
    spread = GX[:, i] - h_ols * GX[:, j]
    try:
        eg = float(coint(GX[:, i], GX[:, j], trend="c", autolag="AIC")[1])
    except Exception:
        eg = np.nan
    # integer hedge
    best_k, best_v = None, np.inf
    for k in int_grid:
        v = var_g[i] - 2 * k * G_g[i, j] + k * k * var_g[j]
        if v < best_v:
            best_v, best_k = v, k
    int_spread = GX[:, i] - best_k * GX[:, j]
    group_pair_rows.append({
        "group_a": g_cols[i], "group_b": g_cols[j],
        "level_corr":      float(G_g[i, j] / np.sqrt(var_g[i] * var_g[j])),
        "ols_h":           float(h_ols),
        "ols_resid_ratio": float(np.sqrt(max(rvar, 0) / var_g[i])),
        "ols_eg_p":        eg,
        "ols_adf_p":       adf_p(spread),
        "ols_half_life":   half_life(spread),
        "int_h":           int(best_k),
        "int_resid_ratio": float(np.sqrt(max(best_v, 0) / var_g[i])),
        "int_adf_p":       adf_p(int_spread),
        "int_half_life":   half_life(int_spread),
    })
group_pair_table = pd.DataFrame(group_pair_rows).sort_values("ols_eg_p").reset_index(drop=True)
group_pair_table.to_csv(OUT_DIR / "group_pair_cointegration.csv", index=False)
print("\n=== Group-vs-group OLS + integer hedge (sorted by EG p) ===")
display(group_pair_table.round(4))

# Plot all 10 group indices
fig = go.Figure()
for col in group_idx_df.columns:
    s = group_idx_df[col]
    fig.add_trace(go.Scatter(x=s.index, y=s.values, mode="lines", name=col, line=dict(width=1.0)))
fig.update_layout(height=500, title="Group composite indices (z-mean)", hovermode="x unified")
fig.show()


## §9 — Pure integer linear combinations (no target / no hedge)

Find pure baskets `c₁·P₁ + c₂·P₂ + ... + cₖ·Pₖ` (k ∈ {2, 3, 4, 5}) that are mean-reverting, **without designating any product as "target"**. This is what an arb-trader looks for: any cheap integer combination of the order book that reverts.

Algorithm per subset `S`:

1. Take the smallest-eigenvalue eigenvector of `G[S,S]` — the unit-norm direction of least variance.
2. Normalize so the largest |component| equals `INT_COEF_MAX`, round, clip to `[INT_COEF_MIN, INT_COEF_MAX]`.
3. Search the 3ᵏ neighbourhood around the rounded vector; pick min variance.
4. Drop trivial / degenerate cases:
   - all coefficients zero
   - only one nonzero coefficient (pure single product, not a basket)
5. Filter by **variance ratio** `c·G[S,S]·c / Σ|cᵢ|²·var(Pᵢ)` — small ratio = the combination cancels much of the individual variance, i.e. genuinely co-integrates.

We restrict the per-subset cost: search at most `MAX_SUBSETS_K[k]` random/cheap subsets per `k` to keep total work bounded. For `k=2,3` we exhaust; for `k=4,5` we sample from candidates whose pairwise |level corr| is high (already shortlisted).


In [ ]:
MAX_SUBSETS_PER_K = {2: math.comb(P, 2),
                     3: math.comb(P, 3),
                     4: 50_000,
                     5: 30_000}

def candidate_subsets(k, max_count, seed=RANDOM_SEED):
    """Return list of subset tuples. Exhaustive for small k, sampled for large."""
    if math.comb(P, k) <= max_count:
        return list(combinations(range(P), k))
    rng = np.random.default_rng(seed + k)
    # Bias toward subsets with high pairwise |corr|
    abs_corr = np.abs(G / np.sqrt(np.outer(var, var)))
    np.fill_diagonal(abs_corr, 0)
    # Build a candidate pool: top-15 partners per node
    top_partners = {i: np.argsort(-abs_corr[i])[:15].tolist() for i in range(P)}
    seen = set()
    out = []
    while len(out) < max_count:
        seed_node = int(rng.integers(P))
        chain = [seed_node]
        for _ in range(k - 1):
            cands = [c for c in top_partners[chain[-1]] if c not in chain]
            if not cands:
                break
            chain.append(int(rng.choice(cands)))
        if len(chain) == k:
            tup = tuple(sorted(chain))
            if tup not in seen:
                seen.add(tup)
                out.append(tup)
    return out

def best_integer_combo(S):
    """Eigenvector → integer rounding → 3^k neighbourhood search.
    Returns (c_int, var_combo, var_ratio). var_ratio = combo_var / Σ cᵢ²·var(Pᵢ).
    """
    Gs = G[np.ix_(S, S)]
    eigvals, eigvecs = np.linalg.eigh(Gs)
    v = eigvecs[:, 0]
    if np.max(np.abs(v)) == 0:
        return None, np.inf, np.inf
    scale = INT_COEF_MAX / np.max(np.abs(v))
    near = np.clip(np.round(v * scale).astype(int), INT_COEF_MIN, INT_COEF_MAX)
    best_c, best_v = None, np.inf
    for d in iproduct([-1, 0, 1], repeat=len(S)):
        c = np.clip(near + np.array(d), INT_COEF_MIN, INT_COEF_MAX)
        if (c == 0).all():
            continue
        if (c != 0).sum() < 2:  # require genuine basket
            continue
        v_combo = c @ Gs @ c
        if v_combo < best_v:
            best_v, best_c = v_combo, c.copy()
    if best_c is None:
        return None, np.inf, np.inf
    norm_var = sum((best_c[i] ** 2) * var[S[i]] for i in range(len(S)))
    var_ratio = float(best_v / max(norm_var, 1e-12))
    return best_c, float(best_v), var_ratio

print("Pure integer combination search ...")
pure_rows = []
for k in [2, 3, 4, 5]:
    if k > K_MAX + 1:  # honour user knob loosely
        pass
    subsets = candidate_subsets(k, MAX_SUBSETS_PER_K.get(k, 50_000))
    t0 = time.time()
    keep = []
    for S in subsets:
        S_l = list(S)
        c_int, v_combo, vr = best_integer_combo(S_l)
        if c_int is None or vr > 0.5:  # require at least 2× variance reduction
            continue
        keep.append((S_l, c_int, v_combo, vr))
    keep.sort(key=lambda r: r[3])
    keep = keep[:300]  # top 300 per k for ADF
    print(f"  k={k}: searched {len(subsets):,} subsets in {time.time()-t0:.1f}s, "
          f"keeping {len(keep)} for ADF")
    for S_l, c_int, v_combo, vr in keep:
        prods  = [columns[s] for s in S_l]
        spread = X[:, S_l] @ c_int.astype(float)
        groups = sorted({PRODUCT_TO_GROUP[p] for p in prods})
        pure_rows.append({
            "k":               k,
            "products":        prods,
            "groups":          groups,
            "n_groups":        len(groups),
            "int_coefs":       [int(x) for x in c_int],
            "basket_str":      fmt_basket(prods, c_int),
            "var_combo":       v_combo,
            "var_ratio":       vr,
            "adf_p":           adf_p(spread),
            "half_life":       half_life(spread),
            "spread_std":      float(np.std(spread)),
        })

pure_table = pd.DataFrame(pure_rows).sort_values(["adf_p", "var_ratio"]).reset_index(drop=True)
pure_table.to_csv(OUT_DIR / "pure_integer_combos.csv", index=False)
print(f"\n=== Top pure integer combinations across all k ===")
display(pure_table.head(TOP_DISPLAY).round(4))
for k in sorted(pure_table["k"].unique()):
    print(f"\n=== Top pure integer combos — k = {k} ===")
    display(pure_table.query("k == @k").head(TOP_DISPLAY).round(4))


## §10 — Cross-group focus

For each ordered pair of groups `(G_target, G_basket)`, find the strongest cross-group basket where the target product comes from `G_target` and every basket product comes from a different group (with at least one from `G_basket`). This produces a 10×10 matrix view of "where the cross-group edges live".


In [ ]:
cross_rows = []
all_int = pd.concat(basket_int.values(), ignore_index=True) if basket_int else pd.DataFrame()
if len(all_int):
    for _, row in all_int.iterrows():
        if not row["cross_group"]:
            continue
        for gb in row["basket_groups"]:
            cross_rows.append({**row.to_dict(), "from_basket_group": gb})
    cross_table = pd.DataFrame(cross_rows)
    if len(cross_table):
        # best per (target_group, from_basket_group)
        best_per_cell = (cross_table.sort_values("int_adf_p")
                         .groupby(["target_group", "from_basket_group"], as_index=False)
                         .first())
        print("\n=== Best integer cross-group basket per (target_group, basket_group) ===")
        display(best_per_cell[
            ["target_group", "from_basket_group", "target", "k",
             "int_basket_str", "int_resid_std_ratio", "int_adf_p", "int_half_life"]
        ].round(4))
        best_per_cell.to_csv(OUT_DIR / "cross_group_best_per_cell.csv", index=False)

        # Heatmap of best int_adf_p per cell
        pivot_p = best_per_cell.pivot_table(index="target_group", columns="from_basket_group",
                                            values="int_adf_p", aggfunc="min")
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(-np.log10(pivot_p.replace(0, np.nan)),
                    annot=True, fmt=".1f", cmap="viridis", ax=ax,
                    cbar_kws={"label": "-log10(adf_p)"})
        ax.set_title("Cross-group basket strength: -log10(ADF p) of best integer basket per cell")
        plt.tight_layout(); plt.show()
else:
    print("No integer baskets produced — re-run §6/§7 first.")


## §11 — Top results dashboard + residual plots

Compile the strongest result from each section into one ranked shortlist and plot the residual time-series (with ±2σ band and z-score) for the top entries.


In [ ]:
def plot_residual(name, series, window=WINDOW_FAST):
    s = pd.Series(series).dropna()
    rm = s.rolling(window).mean()
    rs = s.rolling(window).std()
    z  = (s - rm) / rs
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                        subplot_titles=[f"{name} — residual + ±2σ", "z-score"])
    fig.add_trace(go.Scatter(y=(rm + 2*rs).values, line=dict(width=0), showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(y=(rm - 2*rs).values, line=dict(width=0), fill="tonexty",
                             fillcolor="rgba(255,165,0,0.15)", showlegend=False), row=1, col=1)
    fig.add_trace(go.Scatter(y=s.values, line=dict(width=0.7, color="steelblue"), name="resid"), row=1, col=1)
    fig.add_trace(go.Scatter(y=rm.values, line=dict(width=1.0, dash="dot", color="orange"),
                             name="rolling mean"), row=1, col=1)
    fig.add_trace(go.Scatter(y=z.values, line=dict(width=0.7, color="purple"), name="z"), row=2, col=1)
    fig.update_layout(height=480, hovermode="x unified", showlegend=False)
    fig.show()

dashboard = []

# §4 best pair
if len(pair_table):
    r = pair_table.dropna(subset=["eg_p"]).iloc[0]
    dashboard.append({"section":"§4 best pair (OLS)", "recipe":f"{r['a']} − {r['hedge_ratio']:.3f}·{r['b']}",
                      "p_value": r["eg_p"], "half_life": r["half_life"], "cross_group": r["cross_group"]})

# §5 best integer pair
if len(int_pair_table):
    r = int_pair_table.dropna(subset=["int_adf_p"]).iloc[0]
    sign = "−" if r["int_h"] > 0 else "+"
    dashboard.append({"section":"§5 best integer pair", "recipe":f"{r['a']} {sign} {abs(r['int_h'])}·{r['b']}",
                      "p_value": r["int_adf_p"], "half_life": r["int_half_life"], "cross_group": r["cross_group"]})

# §6/§7 best integer basket per K
for k, df in basket_int.items():
    df_ok = df.dropna(subset=["int_adf_p"])
    if len(df_ok):
        r = df_ok.iloc[0]
        dashboard.append({"section": f"§7 best integer basket K={k}", "recipe": r["int_basket_str"],
                          "p_value": r["int_adf_p"], "half_life": r["int_half_life"],
                          "cross_group": r["cross_group"]})

# §8 group pair best
if len(group_pair_table):
    r = group_pair_table.dropna(subset=["int_adf_p"]).sort_values("int_adf_p").iloc[0]
    dashboard.append({"section":"§8 best group-index integer hedge",
                      "recipe": f"index({r['group_a']}) {'−' if r['int_h']>0 else '+'} {abs(r['int_h'])}·index({r['group_b']})",
                      "p_value": r["int_adf_p"], "half_life": r["int_half_life"], "cross_group": True})

# §9 best pure integer combos per k
if len(pure_table):
    for k in sorted(pure_table["k"].unique()):
        sub = pure_table.query("k == @k").dropna(subset=["adf_p"])
        if len(sub):
            r = sub.iloc[0]
            dashboard.append({"section": f"§9 best pure integer combo k={k}", "recipe": r["basket_str"],
                              "p_value": r["adf_p"], "half_life": r["half_life"],
                              "cross_group": r["n_groups"] > 1})

dashboard_df = pd.DataFrame(dashboard).sort_values("p_value").reset_index(drop=True)
dashboard_df.to_csv(OUT_DIR / "dashboard.csv", index=False)
print("\n=== TOP RESULTS DASHBOARD ===")
display(dashboard_df.round(4))

# Plot top 5 from §6/§7 integer baskets (residual time series)
print("\nResidual plots for top integer baskets:")
plotted = 0
for k in sorted(basket_int.keys()):
    for _, r in basket_int[k].head(2).iterrows():
        ti     = columns.index(r["target"])
        S      = [columns.index(p) for p in r["basket"]]
        coefs  = np.asarray(r["int_coefs"], dtype=float)
        spread = X[:, ti] - X[:, S] @ coefs
        plot_residual(r["int_basket_str"], spread)
        plotted += 1
        if plotted >= 6:
            break
    if plotted >= 6:
        break

# Plot top 3 from pure integer combos
if len(pure_table):
    for _, r in pure_table.head(3).iterrows():
        S      = [columns.index(p) for p in r["products"]]
        coefs  = np.asarray(r["int_coefs"], dtype=float)
        spread = X[:, S] @ coefs
        plot_residual(r["basket_str"], spread)


## §12 — How to read these results

Every CSV is in `combination_search_outputs/`:

- `corr_*.csv` — full ranked Pearson / Spearman pair correlations (level + returns)
- `pair_cointegration.csv` — all 1225 pairs with EG / ADF / half-life
- `pair_integer_hedge.csv` — same pairs after integer hedge fitting
- `basket_continuous_K{k}.csv` — top continuous baskets per K with R² + ADF p
- `basket_integer_K{k}.csv` — top integer baskets per K (the main output)
- `group_indices.csv` — the 10 z-mean group indices
- `group_pair_cointegration.csv` — group-vs-group OLS + integer hedges
- `pure_integer_combos.csv` — pure integer linear combinations from eigenvector search
- `cross_group_best_per_cell.csv` — best integer basket per (target_group, basket_group) cell
- `dashboard.csv` — one-line winners per section

### Triage rules

1. Filter `adf_p < 0.05` first — those are statistically mean-reverting after costs of search.
2. Filter `half_life ∈ [50, 500]` — fast enough to round-trip in a round, slow enough to clear costs.
3. Filter `int_resid_inflation < 1.10` (in §7 / §10) — integer rounding cost should be small.
4. Prefer `cross_group=True` for diversification (independent edges).
5. **Verify with §11's residual plots**: a real edge has many round-trips through the band; one big move that never reverts is selection bias.

### Position-limit reminder

All Round-5 products have position limit **10**. So a basket like `target − 3·A − 2·B` traded at one unit means `±10 target, ∓30 A, ∓20 B` at full size — your effective trading capacity scales inversely with the largest absolute coefficient in the basket. Prefer integer baskets where `max(|cᵢ|) ≤ 3`.

### Re-run knobs

- Bump `K_MAX` to 5 only if you have RAM and patience (≈ 50× slower than 4).
- Tighten `INT_COEF_MAX` to 3 for stricter "cheap" baskets.
- Lower `TOP_N_PER_K` if ADF passes are slow.
- Set `SUBSAMPLE_FOR_ADF=None` for full-precision ADF on the final shortlist.
